In [90]:
#Set-up
import requests
from pathlib import Path
import pandas as pd
import zipfile
%pip install remotezip
from remotezip import RemoteZip
import random
import shutil
import h5py
import numpy as np
from matplotlib import pyplot as plt

Note: you may need to restart the kernel to use updated packages.


In [62]:
#Get dataset
record = "14551288"
url = f"https://zenodo.org/api/records/{record}"
response = requests.get(url)
data = response.json()

#downlaod relevant files
download_dir = Path("data")
download_dir.mkdir(exist_ok=True)

#download annotations zip
file = next(f for f in data["files"] if f["key"] == "Annotations_CSV.zip")
file_url = file["links"] = file["links"]["self"]
file_path = download_dir / file["key"]
with requests.get(file_url, stream=True) as r:
    r.raise_for_status()
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)


In [63]:
#Get semantic annotation file
ann_extract_path = Path("data/annotations")
with zipfile.ZipFile(file_path, "r") as ann_zip:
    ann_zip.extractall(ann_extract_path)


In [65]:
#Download annotation CSV
metadata = pd.read_csv("data/annotations/Annotations_CSV/Semantic annotation/Phase annotation.csv")

#Get video IDs
row = metadata.iloc[0]
video_id = row["Video ID"]

#Get start and end frames of event boundaries
frames = {
    "blebbing_start": int(row["blebbing (start frame)"]),
    "blebbing_end": int(row["blebbing (end frame)"]),
    "disruption_start": int(row["disruption (start frame)"]),
    "disruption_end": int(row["disruption (end frame)"])
}

In [ ]:
#Extract cell cropped files from each cell type
cell_zip_files = ["Dendritics_crops.zip",
    "Eosinophils_crops.zip",
    "Neutrophils_crops.zip"
]

#Select number of files to select per cell type
num_files = 1

#Set randomizer
random.seed(42)

#partially extract from zip files
selected_files = {}

for cell_name in cell_zip_files:
    file = next(f for f in data["files"] if f["key"] == cell_name)
    file_url = file["links"]["self"]

    with RemoteZip(file_url) as z:

        #do not include cell videos that have mutliple cells in each crop
        files = [f for f in z.infolist() if not f.is_dir() and f.filename.count("_cell") == 1]

        print(f"\n{cell_name}: {len(files)} files")

        #Randomly select files
        selected = random.sample(files, min(num_files, len(files)))
        selected_files[cell_name] = [f.filename for f in selected]

        for f in selected:
            print(f.filename, round(f.file_size / 1e6, 2), "MB")



Dendritics_crops.zip: 7 files
Dendritics_crops/den4_cell1.ims 239.99 MB

Eosinophils_crops.zip: 10 files
Eosinophils_crops/EOS2_cell1.ims 213.4 MB

Neutrophils_crops.zip: 22 files
Neutrophils_crops/Neu10_cell1.ims 111.03 MB


In [97]:
#Download selected files. Note takes approximately 1 minute per file download
download_dir = Path("data/cropped_videos")

for cell_name in cell_zip_files:

    file = next(f for f in data["files"] if f["key"] == cell_name)
    file_url = file["links"]["self"]

    # Create folders for each cell type
    cell_type = cell_name.replace("_crops.zip", "")
    output_dir = download_dir / cell_type
    output_dir.mkdir(parents=True, exist_ok=True)

    with RemoteZip(file_url) as z:

        for filename in selected_files[cell_name]:

            output_path = output_dir / filename
            output_path.parent.mkdir(parents=True, exist_ok=True)

            with z.open(filename) as source:
                with open(output_path, "wb") as destination:
                    shutil.copyfileobj(source, destination)

            print("Downloaded:", output_path)

Downloaded: data\cropped_videos\Dendritics\Dendritics_crops\den4_cell1.ims
Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS2_cell1.ims
Downloaded: data\cropped_videos\Neutrophils\Neutrophils_crops\Neu10_cell1.ims


In [98]:
#store video files
crop_dir = Path("data/cropped_videos")
video_files = list(crop_dir.rglob("*.ims"))

In [106]:
#Create frames directory
frames_dir = Path("data/frames")
frames_dir.mkdir(parents=True, exist_ok=True)

#link annotation row to video
for video_path in video_files:

    video_id = video_path.stem.split("_cell")[0]
    cell_num = int(video_path.stem.split("_cell")[1])
    #get video ID
    rows = metadata[
        metadata["Video ID"].str.lower() == video_id.lower()
    ].reset_index(drop=True)

    if cell_num > len(rows):
        print("No matching annotation:", video_path.name)
        continue

    #identify annotation row for that cell
    row = rows.iloc[cell_num - 1]

    #extract frame information 
    frames = {
        "blebbing_start": row["blebbing (start frame)"],
        "blebbing_end": row["blebbing (end frame)"],
        "disruption_start": row["disruption (start frame)"],
        "disruption_end": row["disruption (end frame)"]
    }

    # Extract channel information
    channel = int(row["Channel"].lower().replace("ch", "").split("/")[0]) - 1

    with h5py.File(video_path, "r") as video:

        dataset = video["DataSet/ResolutionLevel 0"]

        for label, frame_num in frames.items():

            # Skip missing annotations
            if pd.isna(frame_num):
                continue

            # Assumes annotation frames start at 1
            t = int(frame_num) - 1

            frame_path = f"TimePoint {t}/Channel {channel}/Data"

            if frame_path not in dataset:
                print("Missing frame:", video_path.name, frame_path)
                continue

            # Read 3D frame
            frame = dataset[frame_path][:]

            # Convert to 2D image
            image_2d = np.mean(frame, axis=0)

            # Save image with video, cell, and stage label
            filename = f"{video_id}_cell{cell_num}_{label}.png"

            plt.imsave(
                frames_dir / filename,
                image_2d,
                cmap="gray"
            )

            print("Saved:", filename)    

Saved: Neu10_cell1_disruption_start.png
Saved: Neu10_cell1_disruption_end.png
Saved: EOS2_cell1_blebbing_start.png
Saved: EOS2_cell1_blebbing_end.png
Saved: EOS2_cell1_disruption_start.png
Saved: EOS2_cell1_disruption_end.png
Saved: den4_cell1_disruption_start.png
Saved: den4_cell1_disruption_end.png
